# Gold Layer: Business Metrics for Reporting

**Data Source:** S3 Silver Delta (s3://sales-pipeline-data-lake-ratnajit/silver/)  
**Destination:** S3 Gold Delta (s3://sales-pipeline-data-lake-ratnajit/gold/)

**Business Metrics:**
- Agent performance: tickets handled, average AHT per agent
- Daily ticket volume trends
- Resolution rates by status and issue type
- Issue type distribution

These aggregated metrics feed the Power BI dashboard for Customer Operations stakeholders.

In [0]:
from pyspark.sql.functions import col, count, avg, sum as _sum, date_format, round as _round

# S3 paths - all data stored in S3
SILVER_S3_PATH = 's3://sales-pipeline-data-lake-ratnajit/silver/customer_support_tickets/'
GOLD_S3_BASE = 's3://sales-pipeline-data-lake-ratnajit/gold/'
GOLD_AGENT_S3 = f'{GOLD_S3_BASE}agent_performance/'
GOLD_DAILY_S3 = f'{GOLD_S3_BASE}daily_ticket_volume/'
GOLD_RESOLUTION_S3 = f'{GOLD_S3_BASE}resolution_rates/'

# Table names for Unity Catalog registration
GOLD_AGENT_PERF = 'workspace.gold.agent_performance'
GOLD_DAILY_VOLUME = 'workspace.gold.daily_ticket_volume'
GOLD_RESOLUTION_RATE = 'workspace.gold.resolution_rates'

print(f'Reading from S3: {SILVER_S3_PATH}')
print(f'Writing to S3: {GOLD_S3_BASE}')

Reading from S3: s3://sales-pipeline-data-lake-ratnajit/silver/customer_support_tickets/
Writing to S3: s3://sales-pipeline-data-lake-ratnajit/gold/


In [0]:
# Read directly from S3 Delta
df_silver = spark.read.format('delta').load(SILVER_S3_PATH)

print(f'Silver records: {df_silver.count()}')
print(f'Date range: {df_silver.agg({"created_date": "min"}).collect()[0][0]} to {df_silver.agg({"created_date": "max"}).collect()[0][0]}')

print('\nSilver schema:')
df_silver.printSchema()

Silver records: 1000
Date range: 2026-07-01 to 2026-09-29

Silver schema:
root
 |-- ticket_id: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- aht_minutes: integer (nullable = true)
 |-- created_date: date (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_url: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _page: integer (nullable = true)



In [0]:
# Agent performance: tickets handled and average AHT per agent
df_agent_perf = df_silver.groupBy('agent_name').agg(
    count('ticket_id').alias('tickets_handled'),
    _round(avg('aht_minutes'), 2).alias('avg_aht_minutes'),
    _sum('aht_minutes').alias('total_aht_minutes')
).orderBy(col('tickets_handled').desc())

print('Agent Performance:')
df_agent_perf.show()

# Write to S3 Delta table
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.gold')

df_agent_perf.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .option('path', GOLD_AGENT_S3) \
    .saveAsTable(GOLD_AGENT_PERF)

print(f'Agent performance written: {df_agent_perf.count()} records')
print(f'Location: {GOLD_AGENT_S3}')

Agent Performance:
+----------+---------------+---------------+-----------------+
|agent_name|tickets_handled|avg_aht_minutes|total_aht_minutes|
+----------+---------------+---------------+-----------------+
|      Neha|            265|          24.97|             6617|
|     Priya|            250|          25.69|             6422|
|     Rahul|            247|          25.64|             6333|
|      Amit|            238|           24.6|             5854|
+----------+---------------+---------------+-----------------+

Agent performance written: 4 records
Location: s3://sales-pipeline-data-lake-ratnajit/gold/agent_performance/


In [0]:
# Daily ticket volume trends
df_daily_volume = df_silver.groupBy('created_date').agg(
    count('ticket_id').alias('total_tickets'),
    count(col('status') == 'Resolved').alias('resolved_tickets'),
    count(col('status') == 'Open').alias('open_tickets'),
    count(col('status') == 'Pending').alias('pending_tickets'),
    _round(avg('aht_minutes'), 2).alias('avg_aht')
).orderBy('created_date')

print('Daily Ticket Volume (last 10 days):')
df_daily_volume.orderBy(col('created_date').desc()).show(10)

# Write to S3 Delta table
df_daily_volume.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .option('path', GOLD_DAILY_S3) \
    .saveAsTable(GOLD_DAILY_VOLUME)

print(f'Daily volume written: {df_daily_volume.count()} records')
print(f'Location: {GOLD_DAILY_S3}')

Daily Ticket Volume (last 10 days):
+------------+-------------+----------------+------------+---------------+-------+
|created_date|total_tickets|resolved_tickets|open_tickets|pending_tickets|avg_aht|
+------------+-------------+----------------+------------+---------------+-------+
|  2026-09-29|            9|               9|           9|              9|   24.0|
|  2026-09-28|           15|              15|          15|             15|   27.4|
|  2026-09-27|           13|              13|          13|             13|  25.92|
|  2026-09-26|           11|              11|          11|             11|  20.55|
|  2026-09-25|           13|              13|          13|             13|  24.23|
|  2026-09-24|            6|               6|           6|              6|   32.0|
|  2026-09-23|           19|              19|          19|             19|  23.79|
|  2026-09-22|           15|              15|          15|             15|  24.13|
|  2026-09-21|           12|              12|      

In [0]:
# Resolution rates by issue type and status
df_resolution = df_silver.groupBy('issue_type', 'status').agg(
    count('ticket_id').alias('ticket_count'),
    _round(avg('aht_minutes'), 2).alias('avg_aht')
).orderBy('issue_type', 'status')

print('Resolution Rates by Issue Type and Status:')
df_resolution.show()

# Calculate resolution percentage per issue type
from pyspark.sql.window import Window
window = Window.partitionBy('issue_type')

df_resolution_pct = df_resolution.withColumn(
    'total_per_issue', _sum('ticket_count').over(window)
).withColumn(
    'percentage', _round((col('ticket_count') / col('total_per_issue')) * 100, 2)
).orderBy('issue_type', col('ticket_count').desc())

print('\nResolution Percentage by Issue Type:')
df_resolution_pct.show()

# Write to S3 Delta table
df_resolution_pct.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .option('path', GOLD_RESOLUTION_S3) \
    .saveAsTable(GOLD_RESOLUTION_RATE)

print(f'Resolution rates written: {df_resolution_pct.count()} records')
print(f'Location: {GOLD_RESOLUTION_S3}')

Resolution Rates by Issue Type and Status:
+----------+--------+------------+-------+
|issue_type|  status|ticket_count|avg_aht|
+----------+--------+------------+-------+
|   Billing|    Open|          74|  22.47|
|   Billing| Pending|          50|  21.92|
|   Billing|Resolved|         189|  22.76|
|     Login|    Open|          66|  17.32|
|     Login| Pending|          48|  17.52|
|     Login|Resolved|         229|  17.76|
| Technical|    Open|          72|  34.64|
| Technical| Pending|          52|  35.37|
| Technical|Resolved|         220|  35.38|
+----------+--------+------------+-------+


Resolution Percentage by Issue Type:
+----------+--------+------------+-------+---------------+----------+
|issue_type|  status|ticket_count|avg_aht|total_per_issue|percentage|
+----------+--------+------------+-------+---------------+----------+
|   Billing|Resolved|         189|  22.76|            313|     60.38|
|   Billing|    Open|          74|  22.47|            313|     23.64|
|   Billi

In [0]:
# Verify all gold tables by reading directly from S3
print('Gold Layer Summary - Reading from S3:')

df_agent = spark.read.format('delta').load(GOLD_AGENT_S3)
print(f'Agent Performance: {df_agent.count()} agents')

df_daily = spark.read.format('delta').load(GOLD_DAILY_S3)
print(f'Daily Volume: {df_daily.count()} days')

df_resolution = spark.read.format('delta').load(GOLD_RESOLUTION_S3)
print(f'Resolution Rates: {df_resolution.count()} combinations')

print('\nTop 3 Agents by Tickets Handled:')
df_agent.orderBy(col('tickets_handled').desc()).show(3)

print('\nRecent Daily Volume:')
df_daily.orderBy(col('created_date').desc()).show(5)

print('\nResolution Rate Summary:')
df_resolution.filter(col('status') == 'Resolved').select('issue_type', 'status', 'ticket_count', 'percentage').orderBy(col('percentage').desc()).show()

Gold Layer Summary - Reading from S3:
Agent Performance: 4 agents
Daily Volume: 91 days
Resolution Rates: 9 combinations

Top 3 Agents by Tickets Handled:
+----------+---------------+---------------+-----------------+
|agent_name|tickets_handled|avg_aht_minutes|total_aht_minutes|
+----------+---------------+---------------+-----------------+
|      Neha|            265|          24.97|             6617|
|     Priya|            250|          25.69|             6422|
|     Rahul|            247|          25.64|             6333|
+----------+---------------+---------------+-----------------+
only showing top 3 rows

Recent Daily Volume:
+------------+-------------+----------------+------------+---------------+-------+
|created_date|total_tickets|resolved_tickets|open_tickets|pending_tickets|avg_aht|
+------------+-------------+----------------+------------+---------------+-------+
|  2026-09-29|            9|               9|           9|              9|   24.0|
|  2026-09-28|           1